# 🏯 百度贴吧爬虫 v3.0 - 客户端 API 直连版

**核心突破：使用贴吧客户端内部 API (POST + MD5签名)，完全绕过网页 403/验证码。**

三通道自动降级: `客户端API` → `Cookie预热+移动端` → `Cookie预热+PC端`

**使用：** 依次运行每个 Cell。在 Cell 2 中修改配置。

## 1️⃣ 安装依赖

In [ ]:
!pip install requests beautifulsoup4 lxml pandas -q
print('✅ 依赖安装完成')

## 2️⃣ 配置参数

In [ ]:
TIEBA_NAME = "三国志战略版"     # 🎯 贴吧名称

# ──────── 爬取模式(三选一) ────────
SCRAPE_MODE = "pages"
SCRAPE_PAGES = 3                # 爬取页数 (每页约30条)

# SCRAPE_MODE = "hours"
# SCRAPE_HOURS = 24

# SCRAPE_MODE = "days"
# SCRAPE_DAYS = 7

# ──────── 高级配置 ────────
INCLUDE_TOP = False
MAX_SCAN_PAGES = 20
DELAY_RANGE = (1.0, 2.5)       # 客户端API限制更宽松,可以适当缩短
SAVE_TO_DRIVE = False
OUTPUT_FILENAME = "tieba_data.json"

print(f"✅ 配置完成: [{TIEBA_NAME}吧] | 模式={SCRAPE_MODE}")

## 3️⃣ 爬虫核心代码

In [ ]:
import hashlib
import json
import random
import re
import time as _time
from datetime import datetime, timedelta
from typing import Optional
from urllib.parse import quote, urlencode

import requests
from bs4 import BeautifulSoup

# ---------------------------------------------------------------------------
# 客户端 API 签名
# ---------------------------------------------------------------------------

SIGN_KEY = "tiebaclient!!!"

def calc_sign(params: dict) -> str:
    """
    计算百度贴吧客户端 API 的签名。
    算法: 将所有参数按 key 排序拼接为 "k1=v1k2=v2..." + SIGN_KEY，取 MD5 大写。
    """
    raw = "".join(f"{k}={params[k]}" for k in sorted(params.keys()))
    raw += SIGN_KEY
    return hashlib.md5(raw.encode("utf-8")).hexdigest().upper()


# ---------------------------------------------------------------------------
# UA 池
# ---------------------------------------------------------------------------

MOBILE_UA_POOL = [
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_4 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Mobile/15E148 Safari/604.1",
    "Mozilla/5.0 (Linux; Android 14; SM-S918B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.6261.64 Mobile Safari/537.36",
    "Mozilla/5.0 (Linux; Android 13; Pixel 7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.6167.143 Mobile Safari/537.36",
    "Mozilla/5.0 (Linux; Android 14; SAMSUNG SM-A546B) AppleWebKit/537.36 (KHTML, like Gecko) SamsungBrowser/23.0 Chrome/115.0.0.0 Mobile Safari/537.36",
]

PC_UA_POOL = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0",
]


# ---------------------------------------------------------------------------
# 时间解析
# ---------------------------------------------------------------------------

def normalize_time(time_str) -> str:
    """统一各种时间格式 → 标准字符串"""
    if isinstance(time_str, (int, float)):
        if time_str > 1000000000:
            try:
                return datetime.fromtimestamp(int(time_str)).strftime("%Y-%m-%d %H:%M:%S")
            except (ValueError, OSError):
                pass
        return str(time_str)

    now = datetime.now()
    s = str(time_str).strip()
    if not s:
        return ""

    if s.isdigit() and len(s) >= 9:
        try:
            return datetime.fromtimestamp(int(s)).strftime("%Y-%m-%d %H:%M:%S")
        except (ValueError, OSError):
            pass

    m = re.match(r"(\d+)\s*分钟前", s)
    if m:
        return (now - timedelta(minutes=int(m.group(1)))).strftime("%Y-%m-%d %H:%M")
    m = re.match(r"(\d+)\s*小时前", s)
    if m:
        return (now - timedelta(hours=int(m.group(1)))).strftime("%Y-%m-%d %H:%M")
    if "刚刚" in s:
        return now.strftime("%Y-%m-%d %H:%M")
    m = re.match(r"(\d{4})-(\d{1,2})-(\d{1,2})\s+(\d{1,2}):(\d{2})", s)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d} {int(m.group(4)):02d}:{m.group(5)}"
    m = re.match(r"^(\d{1,2}):(\d{2})$", s)
    if m:
        return f"{now.strftime('%Y-%m-%d')} {int(m.group(1)):02d}:{m.group(2)}"
    m = re.match(r"昨天\s*(\d{1,2}):(\d{2})", s)
    if m:
        y = now - timedelta(days=1)
        return f"{y.strftime('%Y-%m-%d')} {int(m.group(1)):02d}:{m.group(2)}"
    m = re.match(r"^(\d{1,2})-(\d{1,2})$", s)
    if m:
        month, day = int(m.group(1)), int(m.group(2))
        year = now.year
        if month > now.month or (month == now.month and day > now.day):
            year -= 1
        return f"{year}-{month:02d}-{day:02d}"
    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})$", s)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    m = re.match(r"^(\d{4})-(\d{1,2})$", s)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-01"
    return s


# ---------------------------------------------------------------------------
# 通道1: 客户端 API（核心突破）
# ---------------------------------------------------------------------------

class ClientAPI:
    """
    百度贴吧客户端内部 API。
    直接 POST 到 /c/f/frs/page，带签名参数，返回 JSON。
    完全绕过网页反爬/验证码机制。
    """

    ENDPOINT = "https://tieba.baidu.com/c/f/frs/page"
    # 备用端点
    ENDPOINTS = [
        "https://tieba.baidu.com/c/f/frs/page",
        "http://c.tieba.baidu.com/c/f/frs/page",
    ]

    CLIENT_VERSION = "12.67.1.0"

    @staticmethod
    def fetch(kw: str, page_num: int, rn: int = 30) -> Optional[list[dict]]:
        """
        通过客户端 API 获取帖子列表。

        Args:
            kw: 贴吧名
            page_num: 页码(从0开始)
            rn: 每页条数
        Returns:
            帖子列表 或 None
        """
        params = {
            "_client_version": ClientAPI.CLIENT_VERSION,
            "kw": kw,
            "pn": str(page_num + 1),   # 客户端API页码从1开始
            "rn": str(rn),
        }
        params["sign"] = calc_sign(params)

        headers = {
            "Content-Type": "application/x-www-form-urlencoded",
            "User-Agent": f"bdtb for Android {ClientAPI.CLIENT_VERSION}",
            "Accept-Encoding": "gzip",
            "Connection": "keep-alive",
        }

        for endpoint in ClientAPI.ENDPOINTS:
            try:
                resp = requests.post(
                    endpoint,
                    data=params,
                    headers=headers,
                    timeout=15,
                )

                if resp.status_code != 200:
                    continue

                data = resp.json()

                # 检查返回状态
                if data.get("error_code", "0") != "0" and data.get("error_code", 0) != 0:
                    err_msg = data.get("error_msg", "unknown")
                    print(f"    ⚠️  API 返回错误: {data.get('error_code')} - {err_msg}")
                    continue

                # 解析 thread_list
                thread_list = data.get("thread_list") or []
                if not thread_list:
                    # 有时在 data → thread_list
                    thread_list = (data.get("data") or {}).get("thread_list") or []

                if not thread_list:
                    continue

                return ClientAPI._parse_threads(thread_list)

            except (requests.RequestException, json.JSONDecodeError, KeyError) as e:
                print(f"    ⚠️  API 请求异常: {e}")
                continue

        return None

    @staticmethod
    def _parse_threads(thread_list: list) -> list[dict]:
        threads = []
        for item in thread_list:
            if not isinstance(item, dict):
                continue

            tid = int(item.get("id", 0) or item.get("tid", 0) or 0)
            title = str(item.get("title", "") or "")
            if not title:
                continue

            # 摘要: 可能是字符串或 content 列表
            abstract = ""
            abs_data = item.get("abstract") or item.get("first_post_content") or ""
            if isinstance(abs_data, list):
                abstract = "".join(
                    str(a.get("text", "")) for a in abs_data if isinstance(a, dict)
                )
            elif isinstance(abs_data, str):
                abstract = abs_data

            reply_count = int(item.get("reply_num", 0) or 0)
            is_top = bool(item.get("is_top", 0))

            # 作者
            author_data = item.get("author") or {}
            author_name = str(
                author_data.get("name_show", "")
                or author_data.get("name", "")
                or ""
            ) if isinstance(author_data, dict) else str(author_data)

            # 时间 (通常是 Unix 时间戳)
            last_time = (
                item.get("last_time_int")
                or item.get("create_time")
                or ""
            )

            threads.append({
                "thread_id": tid,
                "title": title,
                "abstract": abstract,
                "reply_count": reply_count,
                "is_top": is_top,
                "url": f"https://tieba.baidu.com/p/{tid}" if tid else "",
                "author": {"name": author_name, "name_id": author_name},
                "last_reply": {
                    "author": str((item.get("last_replyer") or {}).get("name_show", "")),
                    "time": normalize_time(last_time),
                    "time_raw": str(last_time),
                },
                "crawl_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        return threads


# ---------------------------------------------------------------------------
# 通道2/3: Cookie 预热 + 网页解析 (备用)
# ---------------------------------------------------------------------------

class WebScraper:
    """
    Cookie 预热策略: 先访问百度首页/贴吧首页获取 BAIDUID Cookie，
    然后带着 Cookie 去请求具体贴吧页面，绕过 403。
    """

    @staticmethod
    def create_warmed_session(mobile: bool = True) -> requests.Session:
        """创建并预热一个带 BAIDUID Cookie 的 Session"""
        sess = requests.Session()
        ua = random.choice(MOBILE_UA_POOL if mobile else PC_UA_POOL)
        sess.headers.update({
            "User-Agent": ua,
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "zh-CN,zh;q=0.9",
            "Accept-Encoding": "gzip, deflate, br",
            "Connection": "keep-alive",
        })

        # 预热: 访问贴吧首页，让百度设置 BAIDUID Cookie
        warmup_urls = [
            "https://tieba.baidu.com/",
            "https://wappass.baidu.com/",
        ]
        for url in warmup_urls:
            try:
                sess.get(url, timeout=10, allow_redirects=True)
                break
            except requests.RequestException:
                continue

        # 额外等待，让 Cookie 生效
        _time.sleep(0.5)
        return sess

    @staticmethod
    def fetch_mobile(sess: requests.Session, kw: str, page_num: int) -> Optional[list[dict]]:
        """移动端网页抓取"""
        params = {"kw": kw, "pn": page_num * 30, "lp": "5028"}
        url = f"https://tieba.baidu.com/mo/q/m?{urlencode(params)}"

        try:
            resp = sess.get(url, timeout=15, allow_redirects=True)
            resp.encoding = "utf-8"
            if resp.status_code != 200 or len(resp.text) < 3000:
                return None
            return WebScraper._parse_page(resp.text)
        except requests.RequestException:
            return None

    @staticmethod
    def fetch_pc(sess: requests.Session, kw: str, page_num: int) -> Optional[list[dict]]:
        """PC端网页抓取"""
        params = {"kw": kw, "ie": "utf-8", "pn": page_num * 50}
        url = f"https://tieba.baidu.com/f?{urlencode(params)}"

        try:
            resp = sess.get(url, timeout=15, allow_redirects=True)
            resp.encoding = "utf-8"
            if resp.status_code != 200 or len(resp.text) < 3000:
                return None
            return WebScraper._parse_page(resp.text)
        except requests.RequestException:
            return None

    @staticmethod
    def _parse_page(html: str) -> list[dict]:
        """统一解析页面 (尝试 SPA JSON → PC HTML)"""
        soup = BeautifulSoup(html, "lxml")
        threads = []

        # 尝试提取 SPA 内嵌 JSON
        for script in soup.find_all("script"):
            text = script.string or ""
            for pat in [
                r'window\.__INITIAL_DATA__\s*=\s*({.+?})\s*;',
                r'"thread_list"\s*:\s*(\[.+?\])\s*[,}]',
            ]:
                m = re.search(pat, text, re.DOTALL)
                if m:
                    try:
                        data = json.loads(m.group(1))
                        tl = None
                        if isinstance(data, list):
                            tl = data
                        elif isinstance(data, dict):
                            tl = data.get("thread_list") or (data.get("data") or {}).get("thread_list")
                        if tl:
                            return ClientAPI._parse_threads(tl)  # 复用解析逻辑
                    except (json.JSONDecodeError, KeyError):
                        continue

        # PC HTML 解析
        for li in soup.select("li.j_thread_list.clearfix") or soup.select("#thread_list > li"):
            data = {}
            raw = li.get("data-field", "")
            if raw:
                try:
                    data = json.loads(raw)
                except (json.JSONDecodeError, TypeError):
                    pass
            title_tag = li.select_one("a.j_th_tit") or li.select_one(".threadlist_title a")
            if not title_tag:
                continue
            title = title_tag.get_text(strip=True)
            if not title:
                continue
            href = title_tag.get("href", "")
            tid = data.get("id", 0)
            if not tid and href:
                m2 = re.search(r"/p/(\d+)", href)
                if m2:
                    tid = int(m2.group(1))
            abs_tag = li.select_one(".threadlist_abs, .threadlist_abs_onlyline")
            time_tag = li.select_one(".threadlist_reply_date, .is_show_create_time")
            time_raw = time_tag.get_text(strip=True) if time_tag else ""
            if not time_raw:
                ts = data.get("last_time_int", "")
                if ts and str(ts).isdigit() and int(str(ts)) > 1e9:
                    time_raw = str(ts)
            last_author_tag = li.select_one(".j_replyer")
            threads.append({
                "thread_id": tid,
                "title": title,
                "abstract": abs_tag.get_text(strip=True) if abs_tag else "",
                "reply_count": int(data.get("reply_num", 0) or 0),
                "is_top": bool(data.get("is_top", False)) or bool(li.select_one(".icon_top")),
                "url": f"https://tieba.baidu.com/p/{tid}" if tid else "",
                "author": {"name": data.get("author_name", ""), "name_id": data.get("author_name", "")},
                "last_reply": {
                    "author": last_author_tag.get_text(strip=True) if last_author_tag else "",
                    "time": normalize_time(time_raw),
                    "time_raw": time_raw,
                },
                "crawl_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        return threads


# ---------------------------------------------------------------------------
# 主爬虫: 三通道调度
# ---------------------------------------------------------------------------

class TiebaScraper:
    """
    三通道贴吧爬虫:
      1️⃣ 客户端 API (POST+sign) → 最稳定，直接返回JSON
      2️⃣ Cookie预热 + 移动端网页 → 备选
      3️⃣ Cookie预热 + PC端网页 → 兜底
    """

    def __init__(self, kw, delay=(1.0, 2.5)):
        self.kw = kw
        self.delay = delay
        self._fails = 0
        self._web_sessions_ready = False
        self._mobile_sess = None
        self._pc_sess = None

    def _ensure_web_sessions(self):
        """延迟初始化网页 Session (仅在 API 失败时才预热)"""
        if not self._web_sessions_ready:
            print("  🔄 正在预热 Web Session (获取 BAIDUID Cookie)...")
            self._mobile_sess = WebScraper.create_warmed_session(mobile=True)
            self._pc_sess = WebScraper.create_warmed_session(mobile=False)
            self._web_sessions_ready = True
            cookies = dict(self._mobile_sess.cookies)
            baiduid = cookies.get("BAIDUID", "未获取")
            print(f"  🍪 Cookie 预热完成 (BAIDUID={baiduid[:16]}...)")

    def _fetch_page(self, page_num: int) -> Optional[list[dict]]:
        """三通道依次尝试"""

        # === 通道1: 客户端 API ===
        print(f"  📡 [CLIENT API] 第{page_num+1}页")
        result = ClientAPI.fetch(self.kw, page_num)
        if result:
            self._fails = 0
            print(f"    ✅ 获取 {len(result)} 条帖子")
            return result

        print(f"    ⚠️  客户端 API 失败，切换 Web 通道")

        # === 通道2: Cookie预热 + 移动端 ===
        self._ensure_web_sessions()
        print(f"  📡 [MOBILE WEB] 第{page_num+1}页")
        result = WebScraper.fetch_mobile(self._mobile_sess, self.kw, page_num)
        if result:
            self._fails = 0
            print(f"    ✅ 获取 {len(result)} 条帖子")
            return result

        print(f"    ⚠️  移动端网页失败，切换 PC 通道")

        # === 通道3: Cookie预热 + PC端 ===
        print(f"  📡 [PC WEB    ] 第{page_num+1}页")
        result = WebScraper.fetch_pc(self._pc_sess, self.kw, page_num)
        if result:
            self._fails = 0
            print(f"    ✅ 获取 {len(result)} 条帖子")
            return result

        self._fails += 1
        print(f"  ❌ 第{page_num+1}页全部通道失败")
        return None

    def scrape_by_pages(self, num_pages=1, include_top=False) -> list[dict]:
        all_threads, seen = [], set()
        for page in range(num_pages):
            result = self._fetch_page(page)
            if not result:
                if self._fails >= 3:
                    print("🛑 连续失败≥3次，终止")
                    break
                continue
            for t in result:
                if t["thread_id"] in seen:
                    continue
                seen.add(t["thread_id"])
                if not include_top and t["is_top"]:
                    continue
                all_threads.append(t)
            print(f"  📊 第{page+1}/{num_pages}页 → 累计 {len(all_threads)} 条")
            if page < num_pages - 1:
                _time.sleep(random.uniform(*self.delay))
        return all_threads

    def scrape_by_time(self, hours=0, days=0, max_scan_pages=20, include_top=False) -> list[dict]:
        cutoff = datetime.now() - timedelta(hours=hours, days=days)
        print(f"  ⏰ 时间截止: {cutoff.strftime('%Y-%m-%d %H:%M')}")
        all_threads, seen = [], set()
        should_stop = False
        for page in range(max_scan_pages):
            result = self._fetch_page(page)
            if not result:
                if self._fails >= 3:
                    break
                continue
            added = 0
            for t in result:
                tid = t["thread_id"]
                if tid in seen:
                    continue
                seen.add(tid)
                if not include_top and t["is_top"]:
                    continue
                rt = t["last_reply"]["time"]
                if rt:
                    try:
                        for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%d"):
                            try:
                                dt = datetime.strptime(rt, fmt)
                                break
                            except ValueError:
                                continue
                        else:
                            all_threads.append(t); added += 1; continue
                        if dt < cutoff:
                            should_stop = True
                            continue
                    except Exception:
                        pass
                all_threads.append(t)
                added += 1
            print(f"  📊 第{page+1}页: +{added}条, 累计{len(all_threads)}条")
            if should_stop:
                print("  ⏹️ 已到达时间截止范围")
                break
            if page < max_scan_pages - 1:
                _time.sleep(random.uniform(*self.delay))
        return all_threads


print("✅ 爬虫核心代码加载完成 (v3.0 客户端API直连)")

## 4️⃣ 执行爬取

In [ ]:
print(f"\n🚀 开始爬取 [{TIEBA_NAME}吧] ...")
print(f"   通道优先级: 客户端API → Cookie预热+移动端 → Cookie预热+PC端")
print("-" * 55)

scraper = TiebaScraper(kw=TIEBA_NAME, delay=DELAY_RANGE)

if SCRAPE_MODE == "pages":
    threads = scraper.scrape_by_pages(num_pages=SCRAPE_PAGES, include_top=INCLUDE_TOP)
elif SCRAPE_MODE == "hours":
    threads = scraper.scrape_by_time(hours=SCRAPE_HOURS, max_scan_pages=MAX_SCAN_PAGES, include_top=INCLUDE_TOP)
elif SCRAPE_MODE == "days":
    threads = scraper.scrape_by_time(days=SCRAPE_DAYS, max_scan_pages=MAX_SCAN_PAGES, include_top=INCLUDE_TOP)
else:
    threads = []

print("-" * 55)
print(f"🎉 爬取完成！共获取 {len(threads)} 条帖子")

## 5️⃣ 查看结果

In [ ]:
import pandas as pd

if threads:
    flat = []
    for t in threads:
        flat.append({
            "标题": t["title"],
            "回复数": t["reply_count"],
            "作者": t["author"]["name"],
            "最后回复时间": t["last_reply"]["time"],
            "最后回复者": t["last_reply"]["author"],
            "摘要": (t["abstract"][:50] + "...") if len(t["abstract"]) > 50 else t["abstract"],
            "帖子链接": t["url"],
        })
    df = pd.DataFrame(flat)
    print(f"\n📊 共 {len(df)} 条帖子:\n")
    # Colab 中用 display(df)，本地用 print
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))
else:
    print("⚠️ 未获取到数据")

## 6️⃣ 保存数据

In [ ]:
if threads:
    if SAVE_TO_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            output_path = f"/content/drive/MyDrive/{OUTPUT_FILENAME}"
        except ImportError:
            output_path = OUTPUT_FILENAME
    else:
        output_path = OUTPUT_FILENAME

    result = {
        "meta": {
            "forum_name": f"{TIEBA_NAME}吧",
            "forum_url": f"https://tieba.baidu.com/f?kw={quote(TIEBA_NAME)}&ie=utf-8",
            "total_posts": len(threads),
            "crawl_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "scraper_version": "3.0.0-colab",
        },
        "posts": threads,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"✅ 已保存 {len(threads)} 条 → {output_path}")

    if not SAVE_TO_DRIVE:
        try:
            from google.colab import files
            files.download(output_path)
            print("📥 下载已触发")
        except ImportError:
            print(f"(本地环境，文件已保存到: {output_path})")
        except Exception as e:
            print(f"下载触发失败: {e}")


# ============================================================================
# ===== CELL 7 (可选): 数据分析 =====
# ============================================================================

if threads and len(threads) > 5:
    print(f"\n📈 数据概览 - {TIEBA_NAME}吧")
    print("=" * 50)
    print(f"  帖子总数:    {len(threads)}")
    print(f"  总回复数:    {df['回复数'].sum()}")
    print(f"  平均回复:    {df['回复数'].mean():.1f}")
    print(f"  最高回复:    {df['回复数'].max()}")
    print(f"  活跃作者数:  {df['作者'].nunique()}")

    print(f"\n🔥 回复数 TOP 5:")
    top5 = df.nlargest(5, "回复数")[["标题", "回复数", "作者"]].reset_index(drop=True)
    top5.index = top5.index + 1
    try:
        display(top5)
    except NameError:
        print(top5.to_string())

    print(f"\n👤 发帖最多的作者 TOP 5:")
    for name, count in df["作者"].value_counts().head(5).items():
        print(f"  {name}: {count}条")